# Range-dependence of pile driving pulse receptions

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

style_sheet = Path("config/jasa_fig_style.mplstyle")
plt.style.use(style_sheet)

METRICS = {
    "SPL_rms": "$L_{p,\\text{rms}}$\n(dB re 1 $\\mathrm{\\mu Pa}$)",
    "SPL_pk": "$L_{p,\\text{pk}}$\n(dB re 1 $\\mathrm{\\mu Pa}$)",
    "SPL_pkpk": "$L_{p,\\text{pk-pk}}$\n(dB re 1 $\\mathrm{\\mu Pa}$)",
    "SEL": "$L_{E,p}$\n(dB re 1 $\\mathrm{\\mu Pa^2 s}$)",
    "SEL_cum": "cSEL\n(dB re 1 $\\mathrm{\\mu Pa^2 s}$)",
    "t_rise": "$t_\\mathrm{r}$ (s)",
    "t_decay": "$t_\\mathrm{d}$ (s)",
    "kurtosis": "$\\beta_2$",
}

## Load data

In [4]:
fname = Path("results/metrics.csv")
savepath = Path("results")
df = pd.read_csv(fname)
df["range"] = df["range"] / 1e3

In [5]:
df

,index,uid,station,channel,time_init,time_end,unit,SPL_rms,SPL_pk,SPL_pkpk,SEL,t_rise,t_decay,kurtosis,SEL_cum,event_name,event_site,filter,filter_freq,range
0,0,32f63841,SHRU921_Tetra,0,2023-07-02T19:55:13.115226Z,2023-07-02T19:55:14.315302Z,uPa,128.136157,142.264826,146.711360,128.312939,0.564094,0.472954,0.595223,128.312939,A12-B,A12,highpass,10.0,8.048791
1,0,d31bc3a8,SHRU921_Tetra,1,2023-07-02T19:55:13.115226Z,2023-07-02T19:55:14.315302Z,uPa,128.084959,142.006868,146.435775,128.294399,0.567474,0.461792,0.645524,128.294399,A12-B,A12,highpass,10.0,8.048791
2,0,b755c38c,SHRU921_Tetra,2,2023-07-02T19:55:13.115226Z,2023-07-02T19:55:14.315302Z,uPa,127.977584,141.835053,146.245844,128.188685,0.566859,0.461997,0.627887,128.188685,A12-B,A12,highpass,10.0,8.048791
3,0,0e9bfa29,SHRU921_Tetra,3,2023-07-02T19:55:13.115226Z,2023-07-02T19:55:14.315302Z,uPa,128.150181,141.947604,146.437415,128.365557,0.567525,0.460358,0.615232,128.365557,A12-B,A12,highpass,10.0,8.048791
4,0,aa4ff4a5,SHRU905_VLA1,2,2023-07-02T19:55:13.983871Z,2023-07-02T19:55:15.183947Z,uPa,127.701476,141.724027,147.327428,128.312494,0.540336,0.398046,1.691348,128.312494,A12-B,A12,highpass,10.0,9.468552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342266,4353,3272a137,SHRU920_VLA2,3,2023-08-03T17:38:10.258931Z,2023-08-03T17:38:11.459007Z,uPa,135.476895,148.362765,153.913994,136.773215,0.505314,0.296050,2.729077,171.627652,A13,A13,highpass,10.0,11.243340
342267,4354,60c1a43f,SHRU920_VLA2,2,2023-08-03T17:38:12.419468Z,2023-08-03T17:38:13.619544Z,uPa,133.452990,146.229437,152.182405,134.845023,0.547709,0.236195,2.406493,171.305474,A13,A13,highpass,10.0,11.243340
342268,4354,35bb417a,SHRU920_VLA2,3,2023-08-03T17:38:12.419468Z,2023-08-03T17:38:13.619544Z,uPa,133.830393,146.445834,152.424253,135.213342,0.548375,0.237168,2.219444,171.628644,A13,A13,highpass,10.0,11.243340
342269,4354,f13d215b,SHRU920_VLA2,0,2023-08-03T17:38:12.423462Z,2023-08-03T17:38:13.623538Z,uPa,132.427143,145.726257,151.542803,133.768949,0.534704,0.258366,2.710082,170.402925,A13,A13,highpass,10.0,11.243340


## Plot pulse SPL vs. range, indexed by pulse number

In [23]:
def plot_metric_vs_range(df: pd.DataFrame, metric: str) -> plt.Figure:
    fig = plt.figure(figsize=(6, 4))
    ax = fig.add_subplot(111)
    sns.scatterplot(
        data=df, x="range", y=metric, hue="index", palette="rocket", alpha=0.6, ax=ax
    )
    ax.set_xlabel("Range (km)")
    ax.set_ylabel(METRICS[metric])
    ax.legend(title="Pulse index")
    ax.set_title(f"All events and stations, channel {df['channel'].iloc[0]}")
    return fig


# Call the function to plot
metrics = ["SPL_pk", "SPL_pkpk"]
for channel in df["channel"].unique():
    for metric in metrics:
        df_plot = df[df["channel"] == channel]
        fig = plot_metric_vs_range(df_plot, metric=metric)
        fig.savefig(
            savepath / f"{metric}_vs_range_ch{channel + 1}.png",
            dpi=600,
            bbox_inches="tight",
            facecolor="white",
        )
        plt.close()

## Plot pulse SPL vs. pulse index, broken out by event and station

In [24]:
def plot_rel_lineplot(df: pd.DataFrame, metric: str) -> sns.FacetGrid:
    text_size = 7
    style = mpl.rc_params_from_file(style_sheet)
    sns.set_style("whitegrid", style)

    g = sns.relplot(
        data=df,
        x="index",
        y=metric,
        col="station",
        row="event_name",
        kind="line",
        linewidth=2.0,
        zorder=5,
        height=2,
        aspect=2,
        legend=False,
        errorbar=None,
        color="tab:red",
    )

    num_items = len(g.axes_dict.items())
    for i, ((event, station), ax) in enumerate(g.axes_dict.items()):
        sns.lineplot(
            data=df[df["station"] == station],
            x="index",
            y=metric,
            units="event_name",
            estimator=None,
            color=".4",
            linewidth=0.5,
            ax=ax,
        )
        ax.set_xlim(0, df["index"].max())
        ax.grid(True, which="both", linestyle="--", linewidth=0.5)

        try:
            r_sta = df_plot[
                (df_plot["station"] == station) & (df_plot["event_name"] == event)
            ]["range"].values[0]
            t = ax.text(
                0.98,
                0.1,
                f"R = {r_sta:.1f} km",
                ha="right",
                va="center",
                transform=ax.transAxes,
                fontsize=text_size,
            )
            t.set_bbox(dict(facecolor="white", alpha=1.0, edgecolor="none", pad=0.0))
        except IndexError:
            pass

        if i < 3:
            ax.text(
                0.5,
                1.1,
                station.replace("_", "/"),
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=text_size,
            )
        if (i + 1) % 3 == 0:
            ax.text(
                1.1,
                0.5,
                f"Event\n{event}",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=text_size,
            )

        if i == num_items - 3:
            ax.set_xlabel("Pulse index")
            ax.set_ylabel(METRICS[metric])
        else:
            ax.set_xlabel(None)
            ax.set_ylabel(None)

    g.set_titles("")
    g.fig.suptitle(f"Channel {df_plot['channel'].values[0] + 1}", fontsize=8, y=1.03)
    g.figure.set_size_inches(6, 8)
    return g


for channel in range(4):
    df_plot = df[df["channel"] == channel]
    g = plot_rel_lineplot(df_plot, metric="SPL_pk")
    g.savefig(
        savepath / f"SPL_pk_vs_pulse_ch{channel + 1}.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close()